**Run evidence** · task `03b_metric_views` · run `1043694103369384` · result **SUCCESS** · executed 2026-10-19 17:11 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `03_semantics/03b_metric_views`.

# 03b — UC Metric Views (governed business KPIs)

Metric views define **reusable, governed** measures separated from dimensions, so
the same KPI (OTD %, price variance, risk exposure, sourcing coverage) is
computed consistently across Genie, dashboards, and the App. Genie queries these
natively.

We create 4 metric views:
* `mv_sourcing_coverage`  — vendor coverage / multi-sourcing by material & plant.
* `mv_supplier_risk`       — risk exposure by category / flag / country.
* `mv_delivery_performance`— on-time delivery % by supplier / plant / month.
* `mv_price_analytics`     — net price stats by material / vendor.

> Requires DBR / serverless with Metric Views (YAML v1.1) support.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
NS = CFG.namespace


def run(sql):
    spark.sql(sql)
    print("  ✓ created")


## mv_sourcing_coverage — multi-sourcing coverage
Dimensions: plant, material group, material type, valid-source flag.
Measures: sourcing rows, distinct vendors, valid vendors, materials.

In [ ]:
run(f"""
CREATE OR REPLACE VIEW {NS}.mv_sourcing_coverage
WITH METRICS
LANGUAGE YAML
COMMENT 'Sourcing coverage & multi-sourcing KPIs by material, plant, group.'
AS $$
version: 1.1
comment: "Sourcing coverage and multi-sourcing metrics"
source: {NS}.SOURCING
joins:
  - name: mat
    source: {NS}.MATERIAL
    on: source.MATNR = mat.MATNR AND source.WERKS = mat.WERKS
dimensions:
  - name: Plant
    expr: source.WERKS
    comment: "Plant the sourcing record applies to (SAP WERKS)."
  - name: Material
    expr: source.MATNR
    comment: "Material number being sourced (SAP MATNR)."
  - name: Material Group
    expr: mat.MATKL
    comment: "Purchasing material group the material belongs to (SAP MATKL), joined from MATERIAL."
  - name: Material Type
    expr: mat.MTART
    comment: "Material type such as ROH=raw / HALB=semi-finished (SAP MTART), joined from MATERIAL."
  - name: Is Valid Source
    expr: source.IS_VALID_SOURCE
    comment: "True when the source-list entry is currently valid and the vendor is not blocked."
measures:
  - name: Sourcing Records
    expr: COUNT(1)
    comment: "Count of source-list rows (material+plant+vendor combinations)."
  - name: Distinct Vendors
    expr: COUNT(DISTINCT source.LIFNR)
    comment: "Number of distinct vendors appearing in the sourcing records (any status)."
  - name: Valid Vendors
    expr: COUNT(DISTINCT CASE WHEN source.IS_VALID_SOURCE THEN source.LIFNR END)
    comment: "Number of distinct valid, non-blocked vendors — the basis for multi-sourcing."
  - name: Distinct Materials
    expr: COUNT(DISTINCT source.MATNR)
    comment: "Number of distinct materials that have at least one sourcing record."
$$
""")

  ✓ created


## mv_supplier_risk — risk exposure
Dimensions: risk category, delivery-risk flag, vendor country.
Measures: assessments, risky vendors, avg/max risk score.

In [ ]:
run(f"""
CREATE OR REPLACE VIEW {NS}.mv_supplier_risk
WITH METRICS
LANGUAGE YAML
COMMENT 'Supplier risk exposure KPIs by category, flag, country.'
AS $$
version: 1.1
comment: "Supplier risk exposure metrics"
source: {NS}.SUPPLIER_RISK
joins:
  - name: sup
    source: {NS}.SUPPLIER
    on: source.LIFNR = sup.LIFNR
dimensions:
  - name: Risk Category
    expr: source.RISK_CATEGORY
    comment: "Risk classification bucket assigned to the supplier (e.g. financial, delivery, quality)."
  - name: Delivery Risk Flag
    expr: source.DELIVERY_RISK_FLAG
    comment: "'X' when the supplier carries an active delivery-risk flag; blank/NULL otherwise."
  - name: Vendor Country
    expr: sup.LAND1
    comment: "Vendor's country key (SAP LAND1), joined from SUPPLIER."
  - name: Valid From
    expr: source.VALID_FROM
    comment: "Start date of the risk assessment's validity window (time-slice start)."
measures:
  - name: Risk Assessments
    expr: COUNT(1)
    comment: "Count of supplier risk-assessment rows (time-sliced records)."
  - name: Distinct Vendors
    expr: COUNT(DISTINCT source.LIFNR)
    comment: "Number of distinct vendors that have at least one risk assessment."
  - name: Delivery Risk Vendors
    expr: COUNT(DISTINCT CASE WHEN source.DELIVERY_RISK_FLAG = 'X' THEN source.LIFNR END)
    comment: "Number of distinct vendors currently flagged with delivery risk (DELIVERY_RISK_FLAG='X')."
  - name: Avg Risk Score
    expr: AVG(source.RISK_SCORE)
    comment: "Average numeric risk score across the selected risk assessments."
  - name: Max Risk Score
    expr: MAX(source.RISK_SCORE)
    comment: "Highest (worst) numeric risk score across the selected risk assessments."
$$
""")

  ✓ created


## mv_delivery_performance — on-time delivery
Dimensions: supplier, plant (via purchasing org proxy), receipt month.
Measures: deliveries, on-time count, OTD rate.

In [ ]:
run(f"""
CREATE OR REPLACE VIEW {NS}.mv_delivery_performance
WITH METRICS
LANGUAGE YAML
COMMENT 'On-time delivery KPIs by supplier and month.'
AS $$
version: 1.1
comment: "Delivery performance metrics"
source: {NS}.PURCHASE_ORDER
filter: GR_DATE IS NOT NULL
dimensions:
  - name: Supplier
    expr: LIFNR
    comment: "Vendor/supplier that delivered the PO line (SAP LIFNR)."
  - name: Purchasing Org
    expr: EKORG
    comment: "Purchasing organization responsible for the order (SAP EKORG)."
  - name: Receipt Month
    expr: DATE_TRUNC('MONTH', GR_DATE)
    comment: "Calendar month of goods receipt (GR_DATE truncated to month)."
  - name: Order Month
    expr: DATE_TRUNC('MONTH', ORDER_DATE)
    comment: "Calendar month the PO was placed (ORDER_DATE truncated to month)."
measures:
  - name: Deliveries
    expr: COUNT(1)
    comment: "Count of received PO lines (rows where GR_DATE is not null, per the view filter)."
  - name: On Time Deliveries
    expr: SUM(CASE WHEN ON_TIME_FLAG THEN 1 ELSE 0 END)
    comment: "Count of received lines that arrived on or before the requested date (ON_TIME_FLAG=true)."
  - name: OTD Rate
    expr: AVG(CASE WHEN ON_TIME_FLAG THEN 1.0 ELSE 0.0 END)
    comment: "On-time delivery rate = on-time lines / received lines (0..1)."
$$
""")

  ✓ created


## mv_price_analytics — net price analytics
Dimensions: material, supplier, purchasing org.
Measures: PO lines, avg/min/max net price.

In [ ]:
run(f"""
CREATE OR REPLACE VIEW {NS}.mv_price_analytics
WITH METRICS
LANGUAGE YAML
COMMENT 'Net price analytics KPIs by material and vendor.'
AS $$
version: 1.1
comment: "Price analytics metrics"
source: {NS}.PURCHASE_ORDER
dimensions:
  - name: Material
    expr: MATNR
    comment: "Material number the PO line is for (SAP MATNR)."
  - name: Supplier
    expr: LIFNR
    comment: "Vendor/supplier on the PO line (SAP LIFNR)."
  - name: Purchasing Org
    expr: EKORG
    comment: "Purchasing organization that placed the order (SAP EKORG)."
measures:
  - name: PO Lines
    expr: COUNT(1)
    comment: "Count of purchase-order line items in scope."
  - name: Avg Net Price
    expr: AVG(NETPR)
    comment: "Average net unit price across the selected PO lines (SAP NETPR)."
  - name: Min Net Price
    expr: MIN(NETPR)
    comment: "Lowest net unit price observed across the selected PO lines."
  - name: Max Net Price
    expr: MAX(NETPR)
    comment: "Highest net unit price observed across the selected PO lines."
  - name: Total Spend
    expr: SUM(NETPR)
    comment: "Sum of net prices across the selected PO lines (proxy for spend)."
$$
""")

  ✓ created


## Verify & smoke-test the metric views

In [ ]:
for mv in ["mv_sourcing_coverage", "mv_supplier_risk", "mv_delivery_performance", "mv_price_analytics"]:
    print(f"===== {mv} =====")
    spark.sql(f"DESCRIBE {NS}.{mv}").show(truncate=False)

# Sample measure queries.
print("OTD by supplier (sample):")
spark.sql(f"""
  SELECT Supplier, MEASURE(`OTD Rate`) AS otd, MEASURE(`Deliveries`) AS n
  FROM {NS}.mv_delivery_performance GROUP BY ALL ORDER BY otd LIMIT 5
""").show()

print("Delivery-risk vendors by country:")
spark.sql(f"""
  SELECT `Vendor Country`, MEASURE(`Delivery Risk Vendors`) AS risky
  FROM {NS}.mv_supplier_risk GROUP BY ALL ORDER BY risky DESC
""").show()

===== mv_sourcing_coverage =====
+------------------+--------------+-------------------------------------------------------------------------------------+--------+
|col_name          |data_type     |comment                                                                              |metadata|
+------------------+--------------+-------------------------------------------------------------------------------------+--------+
|Plant             |string        |Plant the sourcing record applies to (SAP WERKS).                                    |NULL    |
|Material          |string        |Material number being sourced (SAP MATNR).                                           |NULL    |
|Material Group    |string        |Purchasing material group the material belongs to (SAP MATKL), joined from MATERIAL. |NULL    |
|Material Type     |string        |Material type such as ROH=raw / HALB=semi-finished (SAP MTART), joined from MATERIAL.|NULL    |
|Is Valid Source   |boolean       |True when the s

In [ ]:
dbutils.notebook.exit("03b_metric_views: SUCCESS — 4 metric views created.")

Notebook exited: 03b_metric_views: SUCCESS — 4 metric views created.
